In [ ]:
import pandas as pd
import requests
import time
import geopandas as gpd
from shapely.geometry import Point

df=pd.read_csv("../OPENACTIVE_MERGED.csv")
bounds=gpd.read_file("../boundaryfile.geojson")

def geocode(postcode):
    try:
        r=requests.get(f"https://api.postcodes.io/postcodes/{postcode.replace(' ', '')}")
        data=r.json()
        if data["status"] == 200:
            return data["result"]["latitude"],data["result"]["longitude"]
    except Exception as e:
        print(f"failed for {postcode}: {e}")
    return None,None


def assign_borough(rows):
    pts=[Point(xy) for xy in zip(rows["longitude"],rows["latitude"])]
    gdf=gpd.GeoDataFrame(rows,geometry=pts,crs="EPSG:4326").to_crs(bounds.crs)
    joined=gpd.sjoin(gdf,bounds[["LAD24NM","geometry"]],how="left",predicate="within")
    return joined["LAD24NM"]

In [ ]:

# 3 haringey rows with wrong borough
haringey_ids=["HAR_190","HAR_195","HAR_196"]
for sid in haringey_ids:
    row=df[df["session_id"] == sid]
    if row.empty:
        continue
    pc=row["postcode"].values[0]
    lat,lon=geocode(pc)
    if lat is not None:
        df.loc[df["session_id"] == sid,["latitude","longitude"]]=lat,lon
    time.sleep(0.2)

fixed=df[df["session_id"].isin(haringey_ids)].copy()
df.loc[fixed.index,"borough"]=assign_borough(fixed).values
df.loc[df["session_id"] == "HAR_190","borough"]="Haringey"



In [ ]:
better=df["provider_group"] == "Better"
df.loc[better & df["session_count"].isna(),"session_count"]=1
df.loc[better & df["is_online"].isna(),"is_online"]=False
dupes=df[better & df.duplicated(subset=["latitude","longitude"],keep=False)]
postcodes=dupes["postcode"].dropna().unique()

coords={}
for pc in postcodes:
    coords[pc]=geocode(pc)
    time.sleep(0.15)
for idx in dupes.index:
    pc=df.loc[idx,"postcode"]
    lat,lon=coords.get(pc,(None,None))
    if lat is not None:
        df.loc[idx,["latitude","longitude"]]=lat,lon

fixed2=df.loc[dupes.index].copy()
new_boroughs=assign_borough(fixed2)
df.loc[fixed2.index,"borough"]=new_boroughs.where(new_boroughs.notna(),df.loc[fixed2.index,"borough"])

In [ ]:
df["price_status"]="known"
df.loc[df["is_free"] == True,"price_status"]="free"
df.loc[(df["is_free"] == False) & (df["price_gbp"] == 0) & (df["provider_group"] == "Places Leisure"),"price_status"]="variable"
df.to_csv("../OPENACTIVE_MERGED.csv",index=False)
print("saved OpenActive_Merged.csv:",df.shape)